In [ ]:
import os
import pickle
import base64

print("Setup complete.")

In [ ]:
# RISIKO: Ein Benutzer kann die Eingabe manipulieren, um beliebige Systembefehle auszuführen.
# Beispiel: filename = "my_notes.txt; ls -la" listet den Inhalt des aktuellen Verzeichnisses auf.
# LÖSUNG: Benutzereingaben niemals direkt in Systembefehle einfügen.
# Stattdessen sichere APIs (z.B. `subprocess` mit Listen von Argumenten) verwenden und Eingaben validieren.

filename_input = "my_notes.txt" # Sicherer Input
# filename_input = "my_notes.txt; whoami" # Gefährlicher Input

# FALSCH: Unsichere Verwendung von `os.system` mit Benutzereingaben.
# Der Befehl wird direkt in der Shell ausgeführt.
print(f"--- Führe Befehl aus: 'ls -l {filename_input}' ---")
os.system(f"ls -l {filename_input}")
print("--- Ausführung beendet ---")

In [ ]:
# RISIKO: Das `pickle`-Modul in Python ist nicht sicher gegen manipulierte Daten.
# Das Laden (Deserialisieren) eines speziell präparierten Pickle-Objekts kann
# zur Ausführung von beliebigem Code führen.
# LÖSUNG: Niemals Daten aus einer nicht vertrauenswürdigen Quelle mit `pickle` laden.
# Sicherere Serialisierungsformate wie JSON verwenden.

# Schritt 1: Ein Angreifer erstellt ein bösartiges Pickle-Objekt.
# Dieses Objekt führt beim Laden den Befehl `os.system('echo "PWNED by pickle!"')` aus.
class MaliciousPickle:
    def __reduce__(self):
        # Dieser Befehl wird bei der Deserialisierung ausgeführt
        return (os.system, ('echo "==> PWNED by pickle! Code was executed <=="',))

# Der Angreifer serialisiert das Objekt und kodiert es (z.B. in Base64), um es zu versenden.
malicious_payload = pickle.dumps(MaliciousPickle())
encoded_payload = base64.b64encode(malicious_payload)
print(f"Kodierte bösartige Nutzlast: {encoded_payload}\n")


# Schritt 2: Der Server empfängt die Daten und deserialisiert sie ohne Argwohn.
# FALSCH: Blinde Deserialisierung von externen Daten.
print("Versuche, die (bösartigen) Daten zu laden...")
decoded_payload = base64.b64decode(encoded_payload)
unpickled_data = pickle.loads(decoded_payload) # Hier wird der bösartige Code ausgeführt!

print("\nLaden der Daten abgeschlossen.")

In [ ]:
# RISIKO: Ein sicherheitskritisches Ereignis (ein fehlgeschlagener Admin-Login)
# tritt auf, wird aber im Code abgefangen und ignoriert, ohne einen Log-Eintrag zu erzeugen.
# Dies macht es für Sicherheitsteams unmöglich, Angriffsversuche zu erkennen.
# LÖSUNG: Alle sicherheitsrelevanten Ereignisse (erfolgreiche/fehlgeschlagene Logins,
# Zugriffsverweigerungen etc.) immer protokollieren.

def login(username, password):
    # Simulierter Login-Prozess
    if username == "admin" and password != "correct_password":
        # FALSCH: Der Fehler wird abgefangen, aber nicht protokolliert.
        # Niemand erfährt von dem fehlgeschlagenen Versuch.
        try:
            # Simuliert einen Fehler, der bei einem ungültigen Login auftreten könnte
            raise ValueError("Invalid credentials for admin")
        except ValueError:
            # Still und leise fehlschlagen
            pass
        return False
    return True

print("Versuche, mich als Admin mit falschem Passwort anzumelden...")
login("admin", "wrong_password_123")
print("Login-Funktion ausgeführt. Überprüfe deine Logs... du wirst nichts finden.")